# 🌽 MaizeGuard — GPU Model Training on Google Colab

This notebook trains the **MaizeGuard EfficientNetB3** CNN model on a GPU (T4 / V100 / A100) using enhanced two-stage transfer learning with frozen BatchNorm, label smoothing, and AdamW optimization.

### Pipeline Steps:
1. **Environment Setup & GPU Check**
2. **Clone MaizeGuard Repository**
3. **Download PlantVillage Maize Dataset**
4. **Train Model (Head Training + Deep Fine-Tuning)**
5. **Evaluate on Held-out Test Set**
6. **Export & Calibrate TFLite Models (FP16 & INT8)**
7. **Update `metrics.json` & Download Model Artifacts**

## 1. Check GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone -b dev https://github.com/ahmaddev-codex/maize-disease-detection.git
%cd maize-disease-detection
!pip install -r requirements-colab.txt

## 3. Download & Prepare Dataset

In [ ]:
import os, shutil, kagglehub

data_dir = "data/raw/plantvillage/data"
if not (os.path.exists(data_dir) and any(os.scandir(data_dir))):
    print("Downloading PlantVillage Maize dataset...")
    download_path = kagglehub.dataset_download("smaranjitghose/corn-or-maize-leaf-disease-dataset")
    os.makedirs(os.path.dirname(data_dir), exist_ok=True)
    src_dir = os.path.join(download_path, "data") if os.path.exists(os.path.join(download_path, "data")) else download_path
    shutil.copytree(src_dir, data_dir, dirs_exist_ok=True)
    print("Dataset ready at:", data_dir)
else:
    print("Dataset already exists at:", data_dir)

!ls -la data/raw/plantvillage/data

## 4. Run High-Accuracy Training Pipeline

In [ ]:
# Execute turnkey training script
!python scripts/train_colab.py

## 5. Inspect Evaluation & Metrics

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

with open("models/exports/metrics.json") as f:
    metrics = json.load(f)

print("=== Recorded Test Accuracy across Models ===")
rows = []
for name, entry in metrics["models"].items():
    res = entry.get("python")
    if res:
        row = {"Model": name, "Accuracy (%)": round(res["accuracy"] * 100, 2), "Test Samples": res["n"]}
        row.update({f"F1_{c}": round(v["f1"], 3) for c, v in res["per_class"].items()})
        rows.append(row)

display(pd.DataFrame(rows).set_index("Model"))

if os.path.exists("models/exports/confusion_matrix_phase1.png"):
    display(Image("models/exports/confusion_matrix_phase1.png"))

## 6. Download Model Artifacts

In [ ]:
from google.colab import files

!zip -r maize_model_outputs.zip models/exports/ models/checkpoints/
files.download("maize_model_outputs.zip")
print("Download started! Extract these files into your local repository to update models and metrics.")